# Caritas Westminster Dissertation Analysis

## Research Question

To what extent are social action initiatives aligned with patterns of socioeconomic deprivation across the Diocese of Westminster?

This notebook contains the descriptive, comparative and statistical analysis used for the dissertation. It combines Caritas Westminster Annual Parish Financial Return (APFR) social-action records with church-location data, IMD 2025 deprivation indicators and contextual wellbeing data.

The interactive-map and analytical-dashboard workflow is maintained separately because it uses a narrower spatial analytical population based on successfully matched named project records.

Recorded APFR social-action activity is used as the observable measure of provision. The analysis is observational and focuses on geographical patterns and associations rather than project impact or causal effects. Missing or unmatched records are not assumed to represent zero provision, and deanery-level aggregation is used where appropriate to support more stable interpretation under uneven parish reporting.

## 1. Data Loading and Preparation

The analysis begins by loading the Python libraries used for data management, numerical processing and visualisation.

The 2025 APFR workbook supplied by Caritas Westminster contains separate sheets for volunteers, parish roles, social action and data-checking information. These sheets are loaded separately so that their structure, dimensions and available variables can be reviewed before the social-action data are prepared for analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

BASE_DIR = Path.cwd()

if not (BASE_DIR / "Data").exists():
    candidate = BASE_DIR / "Caritas-Westminster Dissertation"
    if (candidate / "Data").exists():
        BASE_DIR = candidate

DATA_DIR = BASE_DIR / "Data"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Data folder not found. Place the supplied datasets in a folder named 'Data'."
    )

### Loading the 2025 APFR workbook

The file path below identifies the 2025 Annual Parish Financial Return workbook supplied for the project.

In [ ]:
file_path = DATA_DIR / "afr_2025.xlsx"

The workbook contains separate sheets covering volunteers, parish roles, social action initiatives and a checksum used for data checking. Each sheet is loaded separately so that its structure can be reviewed.

In [ ]:
volunteers = pd.read_excel(file_path, sheet_name="volunteers")
roles = pd.read_excel(file_path, sheet_name="roles")
social_action = pd.read_excel(file_path, sheet_name="social action")
sa_checksum = pd.read_excel(file_path, sheet_name="sa_checksum")

In [ ]:
datasets = {
    "volunteers": volunteers,
    "roles": roles,
    "social_action": social_action,
    "sa_checksum": sa_checksum
}

summary = pd.DataFrame({
    "dataset": datasets.keys(),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()]
})

summary

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())

In [ ]:
social_action.head()

## 2. Cleaning and Data-Quality Checks

The 2025 APFR social-action data are cleaned before analysis. Column names and text fields are standardised, blank strings and common missing-value labels are converted to `NaN`, and selected reporting fields are normalised.

Missing values are retained as missing rather than replaced with zero. An unreported beneficiary, volunteer or project value cannot be assumed to represent no activity.

The following cells also examine missingness, duplicate records and the number of unique values in each field before the cleaned dataset is used in subsequent analysis.

In [ ]:
social_action_clean = social_action.copy()

In [ ]:
social_action_clean.columns = (
    social_action_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

In [ ]:
social_action_clean = social_action_clean.replace(
    r"^\s*$",
    np.nan,
    regex=True
)

In [ ]:
text_columns = social_action_clean.select_dtypes(
    include="object"
).columns

for column in text_columns:
    social_action_clean[column] = social_action_clean[column].str.strip()

In [ ]:
social_action_clean = social_action_clean.replace(
    ["nan", "NaN", "NAN", "None", "none", "NA", "N/A", "Other: NA"],
    np.nan
)

In [ ]:
social_action_clean["multi_church"] = (
    social_action_clean["multi_church"]
    .str.strip()
    .str.lower()
)

social_action_clean["consolidated"] = (
    social_action_clean["consolidated"]
    .str.lower()
)

social_action_clean["afr_missing"] = (
    social_action_clean["afr_missing"]
    .str.lower()
)

In [ ]:
social_action_clean.isnull().sum()

In [ ]:
social_action_clean.duplicated().sum()

In [ ]:
social_action_clean.nunique()

In [ ]:
social_action_clean["multi_church"].value_counts(dropna=False)

### Flagging Extreme Beneficiary Values

Two extreme values in the reported people-supported field were identified during data-quality review. The original values are retained in the APFR data but excluded from beneficiary-count analyses because their scale is substantially different from the wider distribution and comparable beneficiary values were not available for the same projects in the 2024 APFR data.

This adjustment applies only to analyses using reported people-supported values. It does not affect APFR record counts, volunteer counts, deprivation measures or the principal deanery-level alignment analysis.

In [ ]:
social_action_clean["people_supported_numeric"] = pd.to_numeric(
    social_action_clean["people_supported"],
    errors="coerce"
)

northfields_flag = (
    social_action_clean["parish"].eq("Northfields")
    & social_action_clean["project"].str.strip().str.lower().eq("svp")
    & social_action_clean["people_supported_numeric"].eq(46083)
)

warm_hub_flag = (
    social_action_clean["parish"].eq("West Drayton and Yiewsley")
    & social_action_clean["project"].str.strip().str.lower().eq("warm hub")
    & social_action_clean["people_supported_numeric"].eq(41974)
)

social_action_clean["people_supported_flagged"] = (
    northfields_flag | warm_hub_flag
)

social_action_clean["people_supported_for_analysis"] = (
    social_action_clean["people_supported_numeric"]
    .mask(social_action_clean["people_supported_flagged"])
)

social_action_clean.loc[
    social_action_clean["people_supported_flagged"],
    [
        "parish",
        "project",
        "people_supported_numeric",
        "num_sa_volunteers",
        "frequency"
    ]
]

### Analytical Populations

The analysis uses related but distinct analytical populations. The complete 2025 APFR social-action sheet is retained for descriptive and missingness analysis. Records containing a project name form the identifiable-project subset, while the spatial and dashboard analysis uses the narrower subset of named records that can be matched reliably to parish geography.

Keeping these populations separate avoids treating unnamed or geographically unmatched records as equivalent to successfully matched project records.

In [ ]:
social_action_clean = social_action_clean.reset_index(drop=True)

social_action_clean["record_id"] = np.arange(
    1,
    len(social_action_clean) + 1
)

named_projects = (
    social_action_clean[
        social_action_clean["project"].notna()
    ]
    .copy()
)

analytical_population_summary = pd.Series({
    "Raw 2025 APFR rows": len(social_action_clean),
    "Records with a project name": len(named_projects),
    "Records without a project name": social_action_clean["project"].isna().sum()
})

analytical_population_summary

In [ ]:
social_action_clean.to_csv(
    "social_action_clean.csv",
    index=False
)

In [ ]:
social_action_clean['category'].value_counts().head(20)

In [ ]:
social_action_clean['frequency'].value_counts().head(20)

In [ ]:
social_action_clean['episcopal_area'].value_counts()

## 3. Exploratory Analysis of Recorded Social Action

This section examines the distribution and characteristics of the 2025 APFR social-action records.

The analysis considers variation across episcopal areas, deaneries and parishes, together with reported social-action categories, frequency, volunteer involvement and people supported.

These results describe recorded APFR activity. They should not be interpreted as complete measures of the scale or effectiveness of social-action provision because reporting completeness varies across records and parishes.

In [ ]:
caritas_red = "#D7263D"
caritas_light = "#F08A94"

In [ ]:
top_area = (
    social_action_clean["episcopal_area"]
    .value_counts()
)

colors = [
    caritas_red if value == top_area.max()
    else caritas_light
    for value in top_area.values
]

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    x=top_area.index,
    y=top_area.values,
    hue=top_area.index,
    palette=colors,
    legend=False,
    ax=ax
)

ax.set_title(
    "Distribution of APFR Social-Action Records by Episcopal Area",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Episcopal Area")
ax.set_ylabel("Number of APFR Records")
ax.tick_params(axis="x", rotation=15)

fig.tight_layout()


plt.show()

In [ ]:
top_deaneries = (
    social_action_clean["deanery"]
    .value_counts()
    .head(10)
)

colors = [
    caritas_red if value == top_deaneries.max()
    else caritas_light
    for value in top_deaneries.values
]

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    x=top_deaneries.values,
    y=top_deaneries.index,
    hue=top_deaneries.index,
    palette=colors,
    legend=False,
    ax=ax
)

ax.set_title(
    "Top 10 Deaneries by Number of APFR Social-Action Records",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Number of APFR Records")
ax.set_ylabel("Deanery")

fig.tight_layout()


plt.show()

In [ ]:
top_parishes = (
    social_action_clean['parish']
    .value_counts()
    .sort_values(ascending=False)
    .head(15)
)

colors = [
    caritas_red if value == top_parishes.max()
    else caritas_light
    for value in top_parishes.values
]

plt.figure(figsize=(9, 5))

plt.barh(
    top_parishes.index,
    top_parishes.values,
    color=colors
)

plt.title(
    'Top 15 Parishes by Number of APFR Social-Action Records',
    fontsize=15,
    fontweight='bold'
)

plt.xlabel('Number of APFR Social-Action Records')
plt.ylabel('Parish')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
category_counts = (
    social_action_clean["category"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
    .str.lower()
    .replace("", np.nan)
    .dropna()
    .value_counts()
)

top_categories = category_counts.head(15)

label_map = {
    "org_svp": "Society of St Vincent de Paul",
    "org_aa": "Alcoholics Anonymous"
}

category_labels = [
    label_map.get(category, category.replace("_", " ").title())
    for category in top_categories.index
]

colors = [
    caritas_red if value == top_categories.max()
    else caritas_light
    for value in top_categories.values
]

plt.figure(figsize=(9, 6))

plt.barh(
    category_labels,
    top_categories.values,
    color=colors
)

plt.title(
    "Most Common Social Action Categories",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Number of Category Mentions")
plt.ylabel("Category")

plt.gca().invert_yaxis()
plt.tight_layout()

plt.show()

### Standardising Reported Project Frequency

The APFR frequency field contains both standard response categories and free-text descriptions. Similar responses are grouped into a smaller set of comparable categories, including daily, weekly, fortnightly, monthly, quarterly, annual, continuous or as-needed, and occasional activity.

This transformation is used to make the descriptive frequency distribution easier to interpret. Frequency is not converted into a numerical weighting of provision.

In [ ]:
frequency_clean = (
    social_action_clean["frequency"]
    .dropna()
    .str.strip()
    .str.lower()
)

frequency_clean = frequency_clean.replace({
    r".*daily.*": "Daily",
    r".*(twice weekly|3 times per week|multiple times per week).*": "Multiple Times per Week",
    r".*weekly.*": "Weekly",
    r".*(fortnightly|every two weeks).*": "Fortnightly",
    r".*(monthly|twice a month).*": "Monthly",
    r".*quarterly.*": "Quarterly",
    r".*(annually|yearly|twice a year).*": "Annually",
    r".*(continuous|throughout the year|according to need|depending on the needs).*": "Continuous / As Needed",
    r".*(occasional|ad hoc).*": "Occasional / Ad Hoc"
}, regex=True)

standard_frequencies = [
    "Daily",
    "Multiple Times per Week",
    "Weekly",
    "Fortnightly",
    "Monthly",
    "Quarterly",
    "Annually",
    "Continuous / As Needed",
    "Occasional / Ad Hoc"
]

frequency_clean = frequency_clean.where(
    frequency_clean.isin(standard_frequencies),
    "Other"
)

top_frequency = frequency_clean.value_counts()

colors = [
    caritas_red if value == top_frequency.max()
    else caritas_light
    for value in top_frequency.values
]

plt.figure(figsize=(9, 5))

plt.barh(
    top_frequency.index,
    top_frequency.values,
    color=colors
)

plt.title(
    "Reported Frequency of APFR Social-Action Records",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Number of APFR Social-Action Records")
plt.ylabel("Frequency")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
multi_church_counts = (
    social_action_clean["multi_church"]
    .dropna()
    .str.strip()
    .str.lower()
    .value_counts()
    .reindex(["yes", "no", "soon"], fill_value=0)
)

multi_church_counts.index = ["Yes", "No", "Soon"]

colors = [
    caritas_light,
    caritas_red,
    "#F5C4C9"
]

plt.figure(figsize=(7, 7))

plt.pie(
    multi_church_counts.values,
    labels=multi_church_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors
)

plt.title(
    "Reported Multi-Church Parish Status of APFR Social-Action Records",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

### Volunteer and Beneficiary Distributions

Reported volunteer and beneficiary values are examined as contextual indicators of project scale.

These variables contain missing values and are highly uneven across projects, so they are not used as the principal measure of provision in the alignment analysis.

For selected visualisations, observations above the 95th percentile are excluded from the plotted display to improve readability. This affects the visualisation only and does not redefine the underlying source data.

In [ ]:
volunteer_data = (
    pd.to_numeric(
        social_action_clean["num_sa_volunteers"],
        errors="coerce"
    )
    .dropna()
)

upper_limit = volunteer_data.quantile(0.95)
volunteer_plot_data = volunteer_data[
    volunteer_data <= upper_limit
]

median_volunteers = volunteer_data.median()
excluded_values = len(volunteer_data) - len(volunteer_plot_data)

plt.figure(figsize=(10, 6))

plt.hist(
    volunteer_plot_data,
    bins=15,
    color=caritas_light,
    edgecolor="white"
)

plt.axvline(
    median_volunteers,
    color=caritas_red,
    linestyle="--",
    linewidth=2,
    label=f"Median: {median_volunteers:.0f}"
)

plt.title(
    "Distribution of Reported Volunteers per APFR Social-Action Record",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Reported Number of Volunteers")
plt.ylabel("Number of APFR Social-Action Records")

plt.legend()
plt.figtext(
    0.5,
    0.01,
    f"Values above the 95th percentile are excluded for readability ({excluded_values} records).",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

In [ ]:
top_supported = (
    social_action_clean
    .dropna(
        subset=[
            "people_supported_for_analysis",
            "project"
        ]
    )
    .query("people_supported_for_analysis > 0")
    .nlargest(
        15,
        "people_supported_for_analysis"
    )
    .copy()
)

top_supported["project_label"] = (
    top_supported["project"].str.strip()
    + " — "
    + top_supported["parish"].str.strip()
)

colors = [
    caritas_red
    if value == top_supported["people_supported_for_analysis"].max()
    else caritas_light
    for value in top_supported["people_supported_for_analysis"]
]

plt.figure(figsize=(11, 6))

bars = plt.barh(
    top_supported["project_label"],
    top_supported["people_supported_for_analysis"],
    color=colors
)

plt.title(
    "Top 15 APFR Project Records by Reported People Supported",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Number of People Supported")
plt.ylabel("")
plt.gca().invert_yaxis()
plt.ticklabel_format(style="plain", axis="x")

for bar, value in zip(
    bars,
    top_supported["people_supported_for_analysis"]
):
    plt.text(
        bar.get_width()
        + top_supported["people_supported_for_analysis"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,.0f}",
        va="center",
        fontsize=9
    )

plt.xlim(
    0,
    top_supported["people_supported_for_analysis"].max() * 1.15
)

plt.figtext(
    0.5,
    0.01,
    "Two unverified extreme beneficiary values are excluded from this descriptive chart.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

In [ ]:
top_volunteers = (
    social_action_clean
    .assign(
        volunteers_numeric=pd.to_numeric(
            social_action_clean["num_sa_volunteers"],
            errors="coerce"
        )
    )
    .dropna(subset=["volunteers_numeric", "project"])
    .query("volunteers_numeric > 0")
    .nlargest(15, "volunteers_numeric")
    .copy()
)

top_volunteers["initiative_label"] = (
    top_volunteers["project"].str.strip()
    + " — "
    + top_volunteers["parish"].str.strip()
)

colors = [
    caritas_red if value == top_volunteers["volunteers_numeric"].max()
    else caritas_light
    for value in top_volunteers["volunteers_numeric"]
]

plt.figure(figsize=(11, 6))

bars = plt.barh(
    top_volunteers["initiative_label"],
    top_volunteers["volunteers_numeric"],
    color=colors
)

plt.title(
    "Top 15 APFR Project Records by Reported Volunteers",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Number of Volunteers")
plt.ylabel("")

plt.gca().invert_yaxis()

for bar, value in zip(
    bars,
    top_volunteers["volunteers_numeric"]
):
    plt.text(
        bar.get_width() + top_volunteers["volunteers_numeric"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,.0f}",
        va="center",
        fontsize=9
    )

plt.xlim(
    0,
    top_volunteers["volunteers_numeric"].max() * 1.15
)

plt.tight_layout()
plt.show()

### Parish-Level Descriptive Aggregation

A parish-level summary is created to compare the number of recorded social-action records with reported volunteer and beneficiary information.

APFR social-action records are counted directly at parish level. Volunteer and people-supported values are summed only where values were reported. The two beneficiary values flagged during data-quality review are excluded from beneficiary totals but retained in the original dataset. The use of `min_count=1` ensures that a parish with entirely missing values remains missing rather than being converted to zero.

These parish summaries are used descriptively and are interpreted cautiously because reporting completeness differs between parishes.

In [ ]:
parish_summary = (
    social_action_clean
    .assign(
        volunteers_numeric=pd.to_numeric(
            social_action_clean["num_sa_volunteers"],
            errors="coerce"
        )
    )
    .groupby("parish", as_index=False)
    .agg(
        apfr_records=("parish", "size"),
        volunteers=(
            "volunteers_numeric",
            lambda values: values.sum(min_count=1)
        ),
        people_supported=(
            "people_supported_for_analysis",
            lambda values: values.sum(min_count=1)
        )
    )
)

parish_summary.head()

In [ ]:
top_parishes = (
    parish_summary
    .nlargest(15, "apfr_records")
    .sort_values("apfr_records", ascending=True)
)

colors = [
    caritas_red if value == top_parishes["apfr_records"].max()
    else caritas_light
    for value in top_parishes["apfr_records"]
]

plt.figure(figsize=(10, 6))

bars = plt.barh(
    top_parishes["parish"],
    top_parishes["apfr_records"],
    color=colors
)

plt.title(
    "Top 15 Parishes by Number of APFR Social-Action Records",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Number of APFR Social-Action Records")
plt.ylabel("")

for bar, value in zip(
    bars,
    top_parishes["apfr_records"]
):
    plt.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.0f}",
        va="center",
        fontsize=9
    )

plt.xlim(
    0,
    top_parishes["apfr_records"].max() * 1.12
)

plt.tight_layout()
plt.show()

In [ ]:
top_parish_volunteers = (
    parish_summary
    .dropna(subset=["volunteers"])
    .query("volunteers > 0")
    .nlargest(15, "volunteers")
    .sort_values("volunteers", ascending=True)
)

colors = [
    caritas_red if value == top_parish_volunteers["volunteers"].max()
    else caritas_light
    for value in top_parish_volunteers["volunteers"]
]

plt.figure(figsize=(10, 6))

bars = plt.barh(
    top_parish_volunteers["parish"],
    top_parish_volunteers["volunteers"],
    color=colors
)

plt.title(
    "Top 15 Parishes by Total Reported Volunteers",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Total Reported Volunteers")
plt.ylabel("")

for bar, value in zip(
    bars,
    top_parish_volunteers["volunteers"]
):
    plt.text(
        bar.get_width() + top_parish_volunteers["volunteers"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,.0f}",
        va="center",
        fontsize=9
    )

plt.xlim(
    0,
    top_parish_volunteers["volunteers"].max() * 1.15
)

plt.tight_layout()
plt.show()

In [ ]:
top_parish_supported = (
    parish_summary
    .dropna(subset=["people_supported"])
    .query("people_supported > 0")
    .nlargest(15, "people_supported")
    .sort_values("people_supported", ascending=True)
)

colors = [
    caritas_red if value == top_parish_supported["people_supported"].max()
    else caritas_light
    for value in top_parish_supported["people_supported"]
]

plt.figure(figsize=(10, 6))

bars = plt.barh(
    top_parish_supported["parish"],
    top_parish_supported["people_supported"],
    color=colors
)

plt.title(
    "Top 15 Parishes by Total Reported People Supported",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Total Reported People Supported")
plt.ylabel("")

for bar, value in zip(
    bars,
    top_parish_supported["people_supported"]
):
    plt.text(
        bar.get_width() + top_parish_supported["people_supported"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:,.0f}",
        va="center",
        fontsize=9
    )

plt.xlim(
    0,
    top_parish_supported["people_supported"].max() * 1.16
)

plt.figtext(
    0.5,
    0.01,
    "Two unverified extreme beneficiary values are excluded from parish beneficiary totals.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## 4. Additional Data Sources

The next stage introduces the additional datasets required to move from descriptive APFR analysis to geographical comparison.

The 2024 APFR data provide a year-on-year comparison with 2025. IMD 2025 provides the main socioeconomic deprivation measures used in the alignment analysis. Caritas Westminster church-location data provide the geographical link between parish records and local deprivation indicators.

The 2024 APFR social-action data are cleaned using comparable principles to the 2025 data before descriptive comparisons between the two years are made.

In [ ]:
afr2024_file = DATA_DIR / "afr_2024.xlsx"
afr2025_file = DATA_DIR / "afr_2025.xlsx"
imd_file = DATA_DIR / "imd_2025.csv"

In [ ]:
imd = pd.read_csv(imd_file)

imd.head()

In [ ]:
volunteers_2024 = pd.read_excel(afr2024_file, sheet_name="volunteers")
roles_2024 = pd.read_excel(afr2024_file, sheet_name="roles")
social_action_2024 = pd.read_excel(afr2024_file, sheet_name="social action")

In [ ]:
comparison_summary = pd.DataFrame({
    "dataset": ["volunteers", "roles", "social_action"],
    "rows_2024": [
        volunteers_2024.shape[0],
        roles_2024.shape[0],
        social_action_2024.shape[0]
    ],
    "rows_2025": [
        volunteers.shape[0],
        roles.shape[0],
        social_action.shape[0]
    ],
    "columns_2024": [
        volunteers_2024.shape[1],
        roles_2024.shape[1],
        social_action_2024.shape[1]
    ],
    "columns_2025": [
        volunteers.shape[1],
        roles.shape[1],
        social_action.shape[1]
    ]
})

comparison_summary

In [ ]:
social_action_2024_clean = social_action_2024.copy()

social_action_2024_clean.columns = (
    social_action_2024_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

social_action_2024_clean = social_action_2024_clean.replace(
    r"^\s*$",
    np.nan,
    regex=True
)

text_columns = social_action_2024_clean.select_dtypes(include="object").columns

for column in text_columns:
    social_action_2024_clean[column] = (
        social_action_2024_clean[column]
        .str.strip()
    )

social_action_2024_clean = social_action_2024_clean.replace(
    ["nan", "NaN", "NAN", "None", "none", "NA", "N/A"],
    np.nan
)

In [ ]:
social_action_2024_clean.isnull().sum()

In [ ]:
social_action_2024_clean.duplicated().sum()

In [ ]:
episcopal_area_comparison = pd.concat(
    [
        social_action_2024_clean["episcopal_area"].value_counts(),
        social_action_clean["episcopal_area"].value_counts()
    ],
    axis=1
).fillna(0).astype(int)

episcopal_area_comparison.columns = ["2024", "2025"]

episcopal_area_comparison

In [ ]:
summary = pd.DataFrame({
    "Year": [2024, 2025],
    "APFR Social-Action Records": [
        len(social_action_2024_clean),
        len(social_action_clean)
    ],
    "Unique Parishes": [
        social_action_2024_clean["parish"].nunique(),
        social_action_clean["parish"].nunique()
    ]
})

summary

In [ ]:
table_3_1 = pd.DataFrame({
    "Measure": [
        "APFR social-action records",
        "Parishes represented in APFR social-action data",
        "Deaneries represented",
        "Episcopal areas represented"
    ],
    "2024": [
        len(social_action_2024_clean),
        social_action_2024_clean["parish"].nunique(),
        social_action_2024_clean["deanery"].nunique(),
        social_action_2024_clean["episcopal_area"].nunique()
    ],
    "2025": [
        len(social_action_clean),
        social_action_clean["parish"].nunique(),
        social_action_clean["deanery"].nunique(),
        social_action_clean["episcopal_area"].nunique()
    ]
})

table_3_1

In [ ]:
imd.shape

## 5. Parish Geography and Deprivation Context

Church-location data provide the geographical link between Caritas Westminster parish records and external deprivation measures.

This section creates a parish-level deprivation lookup for the descriptive APFR analysis. Parish names are standardised and compared across the APFR and church datasets, and identified naming differences are harmonised before the deprivation measures are attached.

The interactive-map and dashboard workflow is maintained separately. It uses official 2021 LSOA boundaries and successfully matched named project records, and therefore represents a narrower spatial analytical population than the descriptive analysis shown here.

In [ ]:
churches = pd.read_csv(
    DATA_DIR / "rcdow_churches.csv"
)

In [ ]:
churches.head()

In [ ]:
churches.shape

In [ ]:
churches_clean = churches.copy()

churches_clean.columns = (
    churches_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

text_columns = churches_clean.select_dtypes(include="object").columns

for column in text_columns:
    churches_clean[column] = churches_clean[column].str.strip()

In [ ]:
social_parishes = set(
    social_action_clean["parish"].dropna().unique()
)

church_parishes = set(
    churches_clean["parish"].dropna().unique()
)

unmatched_parishes = sorted(
    social_parishes - church_parishes
)

parish_match_summary = pd.Series({
    "Social action parishes": len(social_parishes),
    "Church dataset parishes": len(church_parishes),
    "Exact matches": len(social_parishes & church_parishes),
    "Unmatched social action parishes": len(unmatched_parishes)
})

parish_match_summary

In [ ]:
unmatched_parishes

### Linking Parishes to IMD 2025

Following parish-name harmonisation, church locations are linked to IMD 2025 using LSOA identifiers.

Documented LSOA-code corrections are applied where necessary before the final parish-level deprivation lookup is created. IMD score, rank and decile measures are then attached to the relevant church and parish records.

Where a parish is associated with multiple church sites or LSOAs, the deprivation measures are aggregated using the median to create one parish-level deprivation profile.

The resulting merge is checked for missing geographical matches, missing IMD values and unintended duplicate rows before further analysis.

This lookup supports the descriptive analysis below and is distinct from the spatial matching used for the interactive maps and dashboard.

In [ ]:
parish_name_map = {
    "Hemel Hempstead East": "Hemel Hempstead East 1",
    "Hemel West": "Hemel Hempstead West 1",
    "Kilburn (Kilburn West)": "Kilburn West",
    "Kilburn (Kilburn)": "Kilburn",
    "Neasden and Stonebridge (Neasden)": "Neasden",
    "Neasden and Stonebridge (Stonebridge)": "Stonebridge",
    "Paddington North": "Paddington",
    "Stevenage (Bedwell)": "Stevenage Bedwell",
    "Stevenage (Shephall)": "Stevenage Shephall",
    "Stevenage (Transfiguration)": "Stevenage Old Town",
    "Welwyn Garden City (Digswell)": "Welwyn Garden City Digswell",
    "Welwyn Garden City (East)": "Welwyn Garden City East"
}

social_action_matched = social_action_clean.copy()

social_action_matched["parish"] = (
    social_action_matched["parish"]
    .replace(parish_name_map)
)

In [ ]:
imd_clean = imd.copy()

imd_clean.columns = (
    imd_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

imd_clean["lsoa_match"] = (
    imd_clean["lsoa_name_2021"]
    .astype(str)
    .str.upper()
    .str.strip()
)

churches_valid = (
    churches_clean
    .dropna(
        subset=[
            "parish",
            "lsoa",
            "latitude",
            "longitude"
        ]
    )
    .copy()
)

churches_valid["lsoa_match"] = (
    churches_valid["lsoa"]
    .astype(str)
    .str.upper()
    .str.strip()
)

imd_metric_columns = [
    column
    for column in imd_clean.columns
    if any(
        word in column
        for word in ["score", "rank", "decile"]
    )
]

churches_imd = churches_valid.merge(
    imd_clean[
        ["lsoa_match"] + imd_metric_columns
    ],
    on="lsoa_match",
    how="left",
    validate="many_to_one"
)

lsoa_code_corrections = {
    "Wembley 2": "E01033932",
    "Watford": "E01035578",
    "Limehouse": "E01035690",
    "Polish Church 3": "E01034030",
    "Fulham 2": "E01035487",
    "Warwick Street": "E01035716",
    "Poplar": "E01035695",
    "Southall": "E01001367",
    "German Church": "E01035687",
    "Hounslow": "E01034038",
    "Hoxton": "E01001780",
    "Buntingford": "E01035588",
    "Acton West": "E01001274",
    "Hertford": "E01035596",
    "Hayes": "E01034612"
}

postcode_imd_fix = (
    pd.DataFrame(
        lsoa_code_corrections.items(),
        columns=["parish", "lsoa_2021_code"]
    )
    .merge(
        imd_clean[
            ["lsoa_code_2021"] + imd_metric_columns
        ],
        left_on="lsoa_2021_code",
        right_on="lsoa_code_2021",
        how="left",
        validate="many_to_one"
    )
)

churches_imd = churches_imd.merge(
    postcode_imd_fix[
        ["parish"] + imd_metric_columns
    ],
    on="parish",
    how="left",
    suffixes=("", "_postcode"),
    validate="many_to_one"
)

for column in imd_metric_columns:
    churches_imd[column] = churches_imd[column].fillna(
        churches_imd[f"{column}_postcode"]
    )

churches_imd = churches_imd.drop(
    columns=[
        f"{column}_postcode"
        for column in imd_metric_columns
    ]
)
aggregation = {
    "latitude": ("latitude", "mean"),
    "longitude": ("longitude", "mean"),
    "church_sites": ("church", "nunique"),
    "lsoa_count": ("lsoa", "nunique")
}

for column in imd_metric_columns:
    aggregation[column] = (column, "median")

parish_imd_lookup = (
    churches_imd
    .groupby("parish", as_index=False)
    .agg(**aggregation)
)

social_imd = social_action_matched.merge(
    parish_imd_lookup,
    on="parish",
    how="left",
    validate="many_to_one"
)

social_imd["parish_imd_score"] = (
    social_imd[
        "index_of_multiple_deprivation_imd_score"
    ]
)

social_imd["parish_imd_decile"] = (
    social_imd[
        "index_of_multiple_deprivation_imd_decile_where_1_is_most_deprived_10%_of_lsoas"
    ]
)

social_churches = social_imd.copy()

unmatched_location_parishes = (
    social_imd.loc[
        social_imd["latitude"].isna(),
        "parish"
    ]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

unmatched_imd_parishes = (
    social_imd.loc[
        social_imd["latitude"].notna()
        & social_imd[
            "index_of_multiple_deprivation_imd_score"
        ].isna(),
        "parish"
    ]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

merge_validation = pd.Series({
    "APFR records before descriptive merge": len(social_action_matched),
    "APFR records after descriptive merge": len(social_imd),
    "Duplicate parish rows after aggregation":
        parish_imd_lookup["parish"].duplicated().sum(),
    "Parishes without location match":
        len(unmatched_location_parishes),
    "Parishes without IMD match":
        len(unmatched_imd_parishes)
})

print("Social IMD shape:", social_imd.shape)

merge_validation

In [ ]:
final_merge_check = pd.Series({
    "Total APFR records": len(social_imd),
    "Missing locations": social_imd["latitude"].isna().sum(),
    "Missing IMD scores": social_imd[
        "index_of_multiple_deprivation_imd_score"
    ].isna().sum(),
    "Duplicate APFR record rows": social_imd.duplicated().sum()
})

final_merge_check

In [ ]:
social_imd[
    "index_of_multiple_deprivation_imd_score"
].isna().sum()

## 6. Deprivation Patterns in Recorded Social Action

The merged APFR and IMD dataset is used to examine the socioeconomic context in which recorded social-action activity occurs.

IMD scores and deciles are explored descriptively across records and geographical units. Higher IMD scores indicate greater relative deprivation, while IMD decile 1 represents the most deprived 10% of English LSOAs.

This stage identifies where recorded activity is located in relation to deprivation. It does not by itself establish whether the level of recorded provision is proportionate to local need.

The figures in this section use the parish-level deprivation lookup described above rather than the narrower matched named-project population used for the interactive maps and dashboard.

In [ ]:
imd_score_column = "index_of_multiple_deprivation_imd_score"

median_imd_score = social_imd[imd_score_column].median()

plt.figure(figsize=(10, 5))

sns.histplot(
    data=social_imd,
    x=imd_score_column,
    bins=20,
    color=caritas_light,
    edgecolor="white"
)

plt.axvline(
    median_imd_score,
    color=caritas_red,
    linestyle="--",
    linewidth=2,
    label=f"Median: {median_imd_score:.1f}"
)

plt.title(
    "Distribution of IMD Scores Across APFR Social-Action Records",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("IMD Score")
plt.ylabel("Number of APFR Social-Action Records")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
imd_decile_column = (
    "index_of_multiple_deprivation_imd_decile_"
    "where_1_is_most_deprived_10%_of_lsoas"
)

imd_decile_counts = (
    social_imd[imd_decile_column]
    .dropna()
    .astype(int)
    .value_counts()
    .reindex(range(1, 11), fill_value=0)
)

colors = [
    caritas_red if value == imd_decile_counts.max()
    else caritas_light
    for value in imd_decile_counts.values
]

plt.figure(figsize=(10, 5))

bars = plt.bar(
    imd_decile_counts.index,
    imd_decile_counts.values,
    color=colors
)

plt.title(
    "APFR Social-Action Records by IMD Decile",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("IMD Decile (1 = Most Deprived, 10 = Least Deprived)")
plt.ylabel("Number of APFR Social-Action Records")
plt.xticks(range(1, 11))

for bar, value in zip(bars, imd_decile_counts.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{value}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
parish_imd_data = (
    social_imd
    .drop_duplicates(subset="parish")
    .dropna(
        subset=[
            "episcopal_area",
            "index_of_multiple_deprivation_imd_score"
        ]
    )
)

area_imd = (
    parish_imd_data
    .groupby("episcopal_area")[
        "index_of_multiple_deprivation_imd_score"
    ]
    .median()
    .sort_values(ascending=False)
)

colors = [
    caritas_red if value == area_imd.max()
    else caritas_light
    for value in area_imd.values
]

plt.figure(figsize=(9, 5))

bars = plt.barh(
    area_imd.index,
    area_imd.values,
    color=colors
)

plt.title(
    "Median IMD Score by Episcopal Area",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Median IMD Score")
plt.ylabel("")

plt.gca().invert_yaxis()

for bar, value in zip(bars, area_imd.values):
    plt.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}",
        va="center",
        fontsize=9
    )

plt.xlim(0, area_imd.max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
top_deprived_parishes = (
    social_imd[
        [
            "parish",
            "deanery",
            "episcopal_area",
            "index_of_multiple_deprivation_imd_score",
            "index_of_multiple_deprivation_imd_decile_where_1_is_most_deprived_10%_of_lsoas"
        ]
    ]
    .drop_duplicates(subset="parish")
    .dropna(
        subset=[
            "index_of_multiple_deprivation_imd_score"
        ]
    )
    .sort_values(
        "index_of_multiple_deprivation_imd_score",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top_deprived_parishes

In [ ]:
imd_score_column = "index_of_multiple_deprivation_imd_score"

category_imd_data = (
    social_imd[
        [
            "category",
            imd_score_column
        ]
    ]
    .dropna()
    .assign(
        category=lambda data: data["category"].str.split(";")
    )
    .explode("category")
)

category_imd_data["category"] = (
    category_imd_data["category"]
    .str.strip()
    .str.lower()
)

top_category_names = (
    category_imd_data["category"]
    .value_counts()
    .head(10)
    .index
)

category_imd = (
    category_imd_data[
        category_imd_data["category"].isin(top_category_names)
    ]
    .groupby("category")
    .agg(
        median_imd_score=(imd_score_column, "median"),
        category_mentions=("category", "size")
    )
    .sort_values("median_imd_score", ascending=False)
)

label_map = {
    "org_svp": "Society of St Vincent de Paul",
    "org_aa": "Alcoholics Anonymous"
}

category_labels = [
    f"{label_map.get(category, category.replace('_', ' ').title())} (n={count})"
    for category, count in zip(
        category_imd.index,
        category_imd["category_mentions"]
    )
]

colors = [
    caritas_red
    if value == category_imd["median_imd_score"].max()
    else caritas_light
    for value in category_imd["median_imd_score"]
]

plt.figure(figsize=(10, 6))

bars = plt.barh(
    category_labels,
    category_imd["median_imd_score"],
    color=colors
)

plt.title(
    "Median IMD Score by Social Action Category",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Median IMD Score")
plt.ylabel("")

plt.gca().invert_yaxis()

for bar, value in zip(
    bars,
    category_imd["median_imd_score"]
):
    plt.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}",
        va="center",
        fontsize=9
    )

plt.xlim(
    0,
    category_imd["median_imd_score"].max() * 1.15
)

plt.tight_layout()
plt.show()

In [ ]:
volunteer_imd = (
    social_imd[
        [
            "num_sa_volunteers",
            "index_of_multiple_deprivation_imd_score"
        ]
    ]
    .assign(
        num_sa_volunteers=lambda data: pd.to_numeric(
            data["num_sa_volunteers"],
            errors="coerce"
        )
    )
    .dropna()
    .query("num_sa_volunteers >= 0")
)

volunteer_correlation = (
    volunteer_imd[
        [
            "num_sa_volunteers",
            "index_of_multiple_deprivation_imd_score"
        ]
    ]
    .corr(method="spearman")
    .iloc[0, 1]
)

upper_limit = volunteer_imd["num_sa_volunteers"].quantile(0.95)

volunteer_plot_data = volunteer_imd[
    volunteer_imd["num_sa_volunteers"] <= upper_limit
]

excluded_count = len(volunteer_imd) - len(volunteer_plot_data)

plt.figure(figsize=(9, 6))

plt.scatter(
    volunteer_plot_data["num_sa_volunteers"],
    volunteer_plot_data[
        "index_of_multiple_deprivation_imd_score"
    ],
    color=caritas_light,
    edgecolors="white",
    linewidths=0.5,
    alpha=1,
    s=50
)


plt.title(
    "Reported Volunteers and Local Deprivation",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Number of Volunteers")
plt.ylabel("IMD Score")

plt.text(
    0.98,
    0.95,
    f"Spearman correlation: {volunteer_correlation:.2f}",
    transform=plt.gca().transAxes,
    ha="right",
    va="top",
    fontsize=10
)

plt.figtext(
    0.5,
    0.01,
    (
        f"Spearman correlation is calculated using all available non-missing "
        f"records. Values above the 95th percentile are excluded from the "
        f"display only for readability ({excluded_count} records)."
    ),
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


### Deprivation Bands

For descriptive presentation, IMD deciles are grouped into three broader deprivation bands.

High deprivation includes deciles 1–3, medium deprivation includes deciles 4–7, and low deprivation includes deciles 8–10.

These categories are analytical groupings created for this study to summarise the distribution of recorded APFR activity. They are not separate official IMD classifications and are used descriptively rather than as measures of impact.

In [ ]:
social_imd["deprivation_band"] = pd.cut(
    social_imd[
        "index_of_multiple_deprivation_imd_decile_where_1_is_most_deprived_10%_of_lsoas"
    ],
    bins=[0, 3, 7, 10],
    labels=[
        "High Deprivation",
        "Medium Deprivation",
        "Low Deprivation"
    ],
    include_lowest=True,
    ordered=True
)

social_imd["deprivation_band"].value_counts(sort=False)

In [ ]:
band_counts = (
    social_imd["deprivation_band"]
    .value_counts(sort=False)
)

colors = [
    caritas_red,
    caritas_light,
    "#F5C4C9"
]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    band_counts.index,
    band_counts.values,
    color=colors
)

plt.title(
    "Distribution of APFR Social-Action Records by Deprivation Band",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Deprivation Band")
plt.ylabel("Number of APFR Social-Action Records")

for bar, value in zip(bars, band_counts.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 4,
        f"{value}",
        ha="center",
        fontsize=10
    )

plt.ylim(0, band_counts.max() * 1.15)
plt.tight_layout()

plt.show()

In [ ]:
deanery_imd = (
    social_imd[
        [
            "parish",
            "deanery",
            "index_of_multiple_deprivation_imd_score"
        ]
    ]
    .drop_duplicates(subset="parish")
    .dropna(
        subset=[
            "deanery",
            "index_of_multiple_deprivation_imd_score"
        ]
    )
    .groupby("deanery")[
        "index_of_multiple_deprivation_imd_score"
    ]
    .median()
    .sort_values(ascending=False)
    .head(10)
)

colors = [
    caritas_red if value == deanery_imd.max()
    else caritas_light
    for value in deanery_imd.values
]

plt.figure(figsize=(10, 6))

bars = plt.barh(
    deanery_imd.index,
    deanery_imd.values,
    color=colors
)

plt.title(
    "Deaneries with the Highest Median IMD Scores",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Median IMD Score")
plt.ylabel("")

plt.gca().invert_yaxis()

for bar, value in zip(bars, deanery_imd.values):
    plt.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}",
        va="center",
        fontsize=9
    )

plt.xlim(0, deanery_imd.max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
correlation_data = (
    social_imd[
        [
            "num_sa_volunteers",
            "people_supported_for_analysis",
            "index_of_multiple_deprivation_imd_score"
        ]
    ]
    .apply(pd.to_numeric, errors="coerce")
)

correlation_results = pd.Series({
    "Volunteers and IMD score": correlation_data[
        [
            "num_sa_volunteers",
            "index_of_multiple_deprivation_imd_score"
        ]
    ].corr(method="spearman").iloc[0, 1],

    "People supported and IMD score": correlation_data[
        [
            "people_supported_for_analysis",
            "index_of_multiple_deprivation_imd_score"
        ]
    ].corr(method="spearman").iloc[0, 1]
}).round(3)

correlation_results

In [ ]:
volunteer_n = (
    correlation_data[
        [
            "num_sa_volunteers",
            "index_of_multiple_deprivation_imd_score"
        ]
    ]
    .dropna()
    .shape[0]
)

beneficiary_n = (
    correlation_data[
        [
            "people_supported_for_analysis",
            "index_of_multiple_deprivation_imd_score"
        ]
    ]
    .dropna()
    .shape[0]
)

print("Volunteer correlation n =", volunteer_n)
print("People-supported correlation n =", beneficiary_n)

## 7. APFR 2024–2025 Comparison

The 2024 and 2025 APFR datasets are compared to provide context on changes in recorded social-action activity and reporting.

The comparison considers total social-action records, reported volunteers, episcopal-area distributions and deanery-level changes.

Differences between the two years are interpreted as changes in recorded APFR evidence rather than direct evidence of equivalent changes in actual social-action provision, because reporting practices and completeness may also differ between years.

In [ ]:
record_comparison = pd.Series({
    "2024": len(social_action_2024_clean),
    "2025": len(social_action_clean)
})

colors = [caritas_light, caritas_red]

plt.figure(figsize=(7, 5))

bars = plt.bar(
    record_comparison.index,
    record_comparison.values,
    color=colors
)

plt.title(
    "Total APFR Social-Action Records: 2024 vs 2025",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Year")
plt.ylabel("Number of APFR Social-Action Records")

for bar, value in zip(
    bars,
    record_comparison.values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 8,
        f"{value:,.0f}",
        ha="center",
        fontsize=10
    )

plt.ylim(
    0,
    record_comparison.max() * 1.15
)

plt.tight_layout()
plt.show()

In [ ]:
volunteer_comparison = pd.Series({
    "2024": pd.to_numeric(
        social_action_2024_clean["num_sa_volunteers"],
        errors="coerce"
    ).sum(),
    "2025": pd.to_numeric(
        social_action_clean["num_sa_volunteers"],
        errors="coerce"
    ).sum()
})

plt.figure(figsize=(7, 5))

bars = plt.bar(
    volunteer_comparison.index,
    volunteer_comparison.values,
    color=[caritas_light, caritas_red]
)

plt.title(
    "Total Reported Volunteers: 2024 vs 2025",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Year")
plt.ylabel("Reported Volunteers")

for bar, value in zip(bars, volunteer_comparison.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + volunteer_comparison.max() * 0.02,
        f"{value:,.0f}",
        ha="center",
        fontsize=10
    )

plt.ylim(0, volunteer_comparison.max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
area_comparison = pd.concat(
    {
        "2024": social_action_2024_clean[
            "episcopal_area"
        ].value_counts(),
        "2025": social_action_clean[
            "episcopal_area"
        ].value_counts()
    },
    axis=1
).fillna(0).astype(int)

area_comparison = area_comparison.sort_values(
    "2025",
    ascending=False
)

x = np.arange(len(area_comparison))
width = 0.36

plt.figure(figsize=(10, 6))

bars_2024 = plt.bar(
    x - width / 2,
    area_comparison["2024"],
    width,
    label="2024",
    color=caritas_light
)

bars_2025 = plt.bar(
    x + width / 2,
    area_comparison["2025"],
    width,
    label="2025",
    color=caritas_red
)

plt.title(
    "APFR Social-Action Records by Episcopal Area: 2024 and 2025",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Episcopal Area")
plt.ylabel("Number of APFR Social-Action Records")

plt.xticks(
    x,
    area_comparison.index,
    rotation=15,
    ha="right"
)

for bars in [bars_2024, bars_2025]:
    for bar in bars:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 2,
            f"{bar.get_height():.0f}",
            ha="center",
            fontsize=9
        )

plt.legend()

plt.ylim(
    0,
    area_comparison[["2024", "2025"]]
    .to_numpy()
    .max() * 1.15
)

plt.tight_layout()

plt.show()

### Deanery-Level Annual Comparison

Deanery-level changes are shown only where comparable deanery labels are available in both years. A deanery label that appears in one year but not the other is retained as missing rather than converted to zero, because absence of the same label does not establish that no social-action activity occurred.

In [ ]:
deanery_comparison = pd.concat(
    {
        "2024": social_action_2024_clean[
            "deanery"
        ].value_counts(),
        "2025": social_action_clean[
            "deanery"
        ].value_counts()
    },
    axis=1
)

deanery_comparison["Change"] = (
    deanery_comparison["2025"]
    - deanery_comparison["2024"]
)

deanery_comparison = (
    deanery_comparison
    .sort_values(
        "Change",
        ascending=False,
        na_position="last"
    )
)

deanery_comparison.head(15)

## 8. Deanery-Level Alignment Analysis

The main alignment analysis is conducted at deanery level to provide a more stable comparison where parish reporting is uneven.

Recorded provision is represented by the number of APFR social-action records within each deanery. Local need is represented by the median parish IMD 2025 score.

Deaneries are ranked separately by deprivation and recorded activity, and a rank gap is calculated to identify differences between the two rankings. Spearman's rank correlation is also used to assess the overall association between deanery deprivation and recorded APFR activity.

A positive rank gap indicates that a deanery ranks more highly on deprivation than on recorded activity. These differences are treated as indicators for further review rather than evidence of under-provision because recorded activity may also be affected by reporting completeness, local capacity and other contextual factors.

In [ ]:
imd_score_column = (
    "index_of_multiple_deprivation_imd_score"
)

deanery_records = (
    social_imd
    .groupby("deanery")
    .size()
    .rename("apfr_records")
)

deanery_deprivation = (
    social_imd[
        [
            "parish",
            "deanery",
            imd_score_column
        ]
    ]
    .drop_duplicates(subset="parish")
    .dropna(
        subset=[
            "deanery",
            imd_score_column
        ]
    )
    .groupby("deanery")[imd_score_column]
    .median()
    .rename("median_imd_score")
)

deanery_analysis = (
    pd.concat(
        [
            deanery_records,
            deanery_deprivation
        ],
        axis=1
    )
    .dropna()
    .reset_index()
)

deanery_analysis["imd_rank"] = (
    deanery_analysis["median_imd_score"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

deanery_analysis["record_count_rank"] = (
    deanery_analysis["apfr_records"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

deanery_analysis["rank_gap"] = (
    deanery_analysis["record_count_rank"]
    - deanery_analysis["imd_rank"]
)

raw_deanery_spearman = (
    deanery_analysis[
        [
            "median_imd_score",
            "apfr_records"
        ]
    ]
    .corr(method="spearman")
    .iloc[0, 1]
)

print(
    f"Raw deanery Spearman correlation: "
    f"{raw_deanery_spearman:.3f}"
)

deanery_analysis.sort_values(
    "rank_gap",
    ascending=False
).head(10)

In [ ]:
rank_gap_plot = (
    deanery_analysis
    .sort_values(
        "rank_gap",
        ascending=True
    )
)

colors = [
    caritas_red if value > 0
    else caritas_light if value < 0
    else "#BDBDBD"
    for value in rank_gap_plot["rank_gap"]
]

plt.figure(figsize=(10, 8))

bars = plt.barh(
    rank_gap_plot["deanery"],
    rank_gap_plot["rank_gap"],
    color=colors
)

plt.axvline(
    0,
    color="black",
    linewidth=1
)

plt.title(
    "Deanery Deprivation–Activity Rank Gaps",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel(
    "Rank Gap: APFR Record-Count Rank − Deprivation Rank"
)

plt.ylabel("")

maximum_gap = (
    rank_gap_plot["rank_gap"]
    .abs()
    .max()
)

for bar, value in zip(
    bars,
    rank_gap_plot["rank_gap"]
):
    plt.text(
        value + (
            0.2 if value >= 0 else -0.2
        ),
        bar.get_y() + bar.get_height() / 2,
        f"{value:.0f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=9
    )

plt.xlim(
    -maximum_gap * 1.25,
    maximum_gap * 1.25
)

plt.text(
    0.98,
    0.02,
    f"Spearman correlation: {raw_deanery_spearman:.2f}",
    transform=plt.gca().transAxes,
    ha="right",
    fontsize=9
)

plt.figtext(
    0.5,
    0.01,
    (
        "Positive values indicate a higher deprivation rank than raw APFR "
        "record-count rank. These differences are review indicators, "
        "not evidence of under-provision."
    ),
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

In [ ]:
plot_data = (
    deanery_analysis
    .dropna(
        subset=[
            "apfr_records",
        ]
    )
    .copy()
)

spearman_corr = (
    plot_data[
        [
            "median_imd_score",
            "apfr_records"
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[0, 1]
)

fig, ax = plt.subplots(
    figsize=(14, 9)
)

ax.scatter(
    plot_data["median_imd_score"],
    plot_data["apfr_records"],
    s=95,
    color="#D7263D",
    alpha=0.80,
    edgecolors="white",
    linewidth=1,
    zorder=3
)

texts = []
original_points = []

for row_number, (_, row) in enumerate(
    plot_data.iterrows()
):
    x_value = row["median_imd_score"]
    y_value = row["apfr_records"]
    deanery_name = str(
        row["deanery"]
    )

    initial_x_offset = (
        0.40
        if row_number % 2 == 0
        else -0.40
    )

    initial_y_offset = (
        0.65
        if row_number % 3 == 0
        else -0.65
        if row_number % 3 == 1
        else 0.15
    )

    text_object = ax.text(
        x_value + initial_x_offset,
        y_value + initial_y_offset,
        deanery_name,
        fontsize=7.5,
        ha=(
            "left"
            if initial_x_offset > 0
            else "right"
        ),
        va="center",
        color="#222222",
        zorder=5
    )

    texts.append(
        text_object
    )

    original_points.append(
        (
            x_value,
            y_value
        )
    )

ax.set_xlim(
    plot_data[
        "median_imd_score"
    ].min() - 2,
    plot_data[
        "median_imd_score"
    ].max() + 4
)

ax.set_ylim(
    plot_data[
        "apfr_records"
    ].min() - 4,
    plot_data[
        "apfr_records"
    ].max() + 5
)

fig.canvas.draw()

renderer = (
    fig.canvas.get_renderer()
)

inverse_transform = (
    ax.transData.inverted()
)

for iteration in range(250):
    text_boxes = [
        text.get_window_extent(
            renderer=renderer
        ).expanded(
            1.08,
            1.20
        )
        for text in texts
    ]

    pixel_shifts = np.zeros(
        (
            len(texts),
            2
        )
    )

    overlap_found = False

    for first_index in range(
        len(texts)
    ):
        for second_index in range(
            first_index + 1,
            len(texts)
        ):
            first_box = text_boxes[
                first_index
            ]

            second_box = text_boxes[
                second_index
            ]

            horizontal_overlap = min(
                first_box.x1,
                second_box.x1
            ) - max(
                first_box.x0,
                second_box.x0
            )

            vertical_overlap = min(
                first_box.y1,
                second_box.y1
            ) - max(
                first_box.y0,
                second_box.y0
            )

            if (
                horizontal_overlap > 0
                and vertical_overlap > 0
            ):
                overlap_found = True

                first_centre_x = (
                    first_box.x0
                    + first_box.x1
                ) / 2

                second_centre_x = (
                    second_box.x0
                    + second_box.x1
                ) / 2

                first_centre_y = (
                    first_box.y0
                    + first_box.y1
                ) / 2

                second_centre_y = (
                    second_box.y0
                    + second_box.y1
                ) / 2

                if (
                    vertical_overlap
                    <= horizontal_overlap
                ):
                    direction = (
                        1
                        if first_centre_y
                        >= second_centre_y
                        else -1
                    )

                    movement = (
                        vertical_overlap / 2
                        + 2
                    )

                    pixel_shifts[
                        first_index,
                        1
                    ] += (
                        direction
                        * movement
                    )

                    pixel_shifts[
                        second_index,
                        1
                    ] -= (
                        direction
                        * movement
                    )

                else:
                    direction = (
                        1
                        if first_centre_x
                        >= second_centre_x
                        else -1
                    )

                    movement = (
                        horizontal_overlap / 2
                        + 2
                    )

                    pixel_shifts[
                        first_index,
                        0
                    ] += (
                        direction
                        * movement
                    )

                    pixel_shifts[
                        second_index,
                        0
                    ] -= (
                        direction
                        * movement
                    )

    if not overlap_found:
        break

    for text_index, text in enumerate(
        texts
    ):
        current_position = (
            text.get_position()
        )

        current_pixel_position = (
            ax.transData.transform(
                current_position
            )
        )

        updated_pixel_position = (
            current_pixel_position
            + pixel_shifts[
                text_index
            ]
        )

        updated_data_position = (
            inverse_transform.transform(
                updated_pixel_position
            )
        )

        text.set_position(
            updated_data_position
        )

    fig.canvas.draw()

    renderer = (
        fig.canvas.get_renderer()
    )

for (
    original_point,
    text_object
) in zip(
    original_points,
    texts
):
    text_x, text_y = (
        text_object.get_position()
    )

    ax.plot(
        [
            original_point[0],
            text_x
        ],
        [
            original_point[1],
            text_y
        ],
        color="#aaaaaa",
        linewidth=0.45,
        alpha=0.60,
        zorder=2
    )

ax.text(
    0.975,
    0.955,
    (
        "Spearman correlation: "
        f"{spearman_corr:.2f}"
    ),
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox={
        "boxstyle": "round,pad=0.40",
        "facecolor": "white",
        "edgecolor": "#777777",
        "linewidth": 0.8,
        "alpha": 0.96
    },
    zorder=6
)

ax.set_title(
    "Deanery Deprivation and Raw APFR Record Counts",
    fontsize=16,
    fontweight="bold",
    pad=16
)

ax.set_xlabel(
    "Median IMD 2025 Score",
    fontsize=11
)

ax.set_ylabel(
    "Number of APFR Social-Action Records",
    fontsize=11
)

ax.grid(
    alpha=0.15,
    linewidth=0.6
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#444444")
    spine.set_linewidth(0.9)

plt.tight_layout()

plt.show()

## 9. Contextual Wellbeing and Community Indicators

Local Authority District-level wellbeing data for 2023–24 and 2024–25 are included to provide wider contextual information around the spatial analysis.

The datasets contain indicators relating to life satisfaction, happiness, anxiety, feelings that life is worthwhile, loneliness and economic conditions.

These measures are used contextually only. They are not incorporated into the parish- or deanery-level alignment measures because they are available at a broader geographical level and capture different dimensions of local conditions.

In [ ]:
outcomes_2024 = pd.read_csv(
    DATA_DIR / "outcomes_2023_24.csv"
)

outcomes_2025 = pd.read_csv(
    DATA_DIR / "outcomes_2024_25.csv"
)

outcomes_summary = pd.DataFrame({
    "Dataset": ["2023–24", "2024–25"],
    "Rows": [
        len(outcomes_2024),
        len(outcomes_2025)
    ],
    "Columns": [
        outcomes_2024.shape[1],
        outcomes_2025.shape[1]
    ]
})

outcomes_summary

In [ ]:
outcomes_compare = outcomes_2024.merge(
    outcomes_2025,
    on="LAD",
    how="inner",
    suffixes=("_2024", "_2025"),
    validate="one_to_one"
)

change_columns = {
    "life_sat_change": "life_sat_mean",
    "happy_change": "happy_mean",
    "anxiety_change": "anxiety_mean",
    "worthwhile_change": "worthwhile_mean"
}

for change_column, metric in change_columns.items():
    outcomes_compare[change_column] = (
        outcomes_compare[f"{metric}_2025"]
        - outcomes_compare[f"{metric}_2024"]
    )

outcomes_compare.shape

In [ ]:
top_life_satisfaction_improvements = (
    outcomes_compare[
        [
            "LAD",
            "life_sat_mean_2024",
            "life_sat_mean_2025",
            "life_sat_change"
        ]
    ]
    .sort_values(
        "life_sat_change",
        ascending=False
    )
    .head(10)
)

top_life_satisfaction_improvements

### Changes in Wellbeing Indicators

Changes between 2023–24 and 2024–25 are calculated for life satisfaction, happiness, anxiety and feelings that life is worthwhile.

The visualisation focuses on Local Authority Districts with the largest combined absolute changes across these measures to make variation easier to inspect.

The changes are descriptive contextual evidence and are not attributed to Caritas Westminster activity.

In [ ]:
wellbeing_changes = (
    outcomes_compare[
        [
            "LAD",
            "life_sat_change",
            "happy_change",
            "anxiety_change",
            "worthwhile_change"
        ]
    ]
    .set_index("LAD")
)

wellbeing_changes["overall_change"] = (
    wellbeing_changes["life_sat_change"].abs()
    + wellbeing_changes["happy_change"].abs()
    + wellbeing_changes["anxiety_change"].abs()
    + wellbeing_changes["worthwhile_change"].abs()
)

wellbeing_plot = (
    wellbeing_changes
    .nlargest(10, "overall_change")
    .drop(columns="overall_change")
)

wellbeing_plot = wellbeing_plot.rename(
    columns={
        "life_sat_change": "Life Satisfaction",
        "happy_change": "Happiness",
        "anxiety_change": "Anxiety",
        "worthwhile_change": "Worthwhile"
    }
)

ax = wellbeing_plot.plot(
    kind="bar",
    figsize=(12, 6),
    color=[
        caritas_red,
        caritas_light,
        "#F5C4C9",
        "#B85A66"
    ]
)

plt.axhline(
    0,
    color="black",
    linewidth=1
)

plt.title(
    "Changes in Selected Wellbeing Indicators by Local Authority District",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Local Authority District")
plt.ylabel("Change from 2023–24 to 2024–25")

plt.xticks(
    rotation=35,
    ha="right"
)

plt.legend(
    title="Wellbeing Measure",
    frameon=False
)

plt.tight_layout()

plt.show()

In [ ]:
[
    column
    for column in outcomes_compare.columns
    if any(
        term in column.lower()
        for term in ["lon", "gdhi", "disposable"]
    )
]

### Loneliness

Frequent loneliness is compared between 2023–24 and 2024–25 at Local Authority District level.

Where necessary, the values are converted to percentage-point changes for consistent presentation. Positive changes indicate an increase in the proportion of respondents reporting that they feel lonely often or always.

Loneliness is used as contextual evidence and is not included in the parish- or deanery-level alignment calculations.

In [ ]:
import re

def normalise_column(name):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(name).lower()
    ).strip("_")

column_lookup = {
    normalise_column(column): column
    for column in outcomes_compare.columns
}

loneliness_2024_column = "lon_oft_prop_2024"
loneliness_2025_column = "lon_oft_prop_2025"

loneliness_plot = outcomes_compare[
    [
        "LAD",
        loneliness_2024_column,
        loneliness_2025_column
    ]
].copy()

loneliness_plot["2023–24"] = pd.to_numeric(
    loneliness_plot[loneliness_2024_column],
    errors="coerce"
)

loneliness_plot["2024–25"] = pd.to_numeric(
    loneliness_plot[loneliness_2025_column],
    errors="coerce"
)

loneliness_plot = loneliness_plot.dropna(
    subset=["2023–24", "2024–25"]
)

largest_value = loneliness_plot[
    ["2023–24", "2024–25"]
].max().max()

scale = 100 if largest_value <= 1 else 1

loneliness_plot["change"] = (
    loneliness_plot["2024–25"]
    - loneliness_plot["2023–24"]
) * scale

loneliness_plot = loneliness_plot.sort_values(
    "change",
    ascending=True
)

colors = [
    caritas_red if value > 0
    else caritas_light
    for value in loneliness_plot["change"]
]

plt.figure(figsize=(10, 9))

bars = plt.barh(
    loneliness_plot["LAD"],
    loneliness_plot["change"],
    color=colors
)

plt.axvline(
    0,
    color="black",
    linewidth=1
)

plt.title(
    "Change in Frequent Loneliness by Local Authority District",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel(
    "Percentage-Point Change from 2023–24 to 2024–25"
)

plt.ylabel("")

maximum_change = max(
    loneliness_plot["change"].abs().max(),
    0.1
)

for bar, value in zip(
    bars,
    loneliness_plot["change"]
):
    plt.text(
        value + (
            maximum_change * 0.025
            if value >= 0
            else -maximum_change * 0.025
        ),
        bar.get_y() + bar.get_height() / 2,
        f"{value:+.1f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=8
    )

plt.xlim(
    -maximum_change * 1.25,
    maximum_change * 1.25
)

plt.figtext(
    0.5,
    0.01,
    "Positive values indicate an increase in the proportion reporting "
    "feeling lonely often or always.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.04, 1, 1])


plt.show()

### Economic Context: GDHI per Head

Gross Disposable Household Income per head for 2022 is included as an additional indicator of local economic context.

The values are examined at Local Authority District level and are used to illustrate differences in wider economic conditions across the study area.

Because GDHI is available at a broader geographical level than the parish and deanery alignment measures, it is not included in the alignment calculations.

In [ ]:
gdhi_columns = [
    (normalised, original)
    for normalised, original in column_lookup.items()
    if "gdhi" in normalised
    or "gross_disposable_household_income" in normalised
]

if not gdhi_columns:
    raise KeyError(
        "The GDHI column could not be identified. "
        "Check the column names printed in the first cell."
    )

gdhi_column = next(
    (
        original
        for normalised, original in gdhi_columns
        if normalised.endswith("2025")
    ),
    gdhi_columns[-1][1]
)

gdhi_plot = outcomes_compare[
    ["LAD", gdhi_column]
].copy()

gdhi_plot["gdhi_per_head"] = pd.to_numeric(
    gdhi_plot[gdhi_column]
    .astype(str)
    .str.replace("£", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

gdhi_plot = (
    gdhi_plot
    .dropna(subset=["gdhi_per_head"])
    .query("gdhi_per_head > 0")
    .sort_values(
        "gdhi_per_head",
        ascending=False
    )
)

lowest_value = gdhi_plot["gdhi_per_head"].min()

colors = [
    caritas_red if value == lowest_value
    else caritas_light
    for value in gdhi_plot["gdhi_per_head"]
]

plt.figure(figsize=(10, 9))

bars = plt.barh(
    gdhi_plot["LAD"],
    gdhi_plot["gdhi_per_head"],
    color=colors
)

plt.title(
    "Gross Disposable Household Income per Head (2022) by Local Authority District",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("GDHI per Head (£, 2022)")

plt.ylabel("")

plt.gca().invert_yaxis()

for bar, value in zip(
    bars,
    gdhi_plot["gdhi_per_head"]
):
    plt.text(
        bar.get_width()
        + gdhi_plot["gdhi_per_head"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"£{value:,.0f}",
        va="center",
        fontsize=8
    )

plt.xlim(
    0,
    gdhi_plot["gdhi_per_head"].max() * 1.18
)

plt.figtext(
    0.5,
    0.01,
    "GDHI is presented as local authority-level economic context "
    "and is not included in the parish or deanery alignment measures.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.04, 1, 1])


plt.show()

## 10. Conceptual Interpretation

The final framework summarises the distinction between estimated local need, actual social-action provision, recorded social-action provision and the spatial alignment observed in the analysis.

Overall and domain-level IMD measures provide evidence about estimated local need, while wellbeing and wider socioeconomic indicators are used as contextual evidence only.

The APFR data capture recorded social-action activity rather than every aspect of actual provision. Reporting completeness and data quality therefore influence what can be observed in the dataset.

The spatial analysis, correlation measures, ranking approaches, interactive maps and analytical dashboard should consequently be interpreted as decision-support tools for identifying patterns and areas for further investigation rather than as definitive measures of parish performance or project impact.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(7.5, 8.5))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

border_colour = "#b00020"
fill_colour = "#fff7f8"
text_colour = "#222222"

def add_framework_box(y, heading, content):
    box = FancyBboxPatch(
        (0.12, y),
        0.76,
        0.13,
        boxstyle="round,pad=0.01,rounding_size=0.014",
        linewidth=1.8,
        edgecolor=border_colour,
        facecolor=fill_colour
    )

    ax.add_patch(box)

    ax.text(
        0.50,
        y + 0.092,
        heading,
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        color=border_colour
    )

    ax.text(
        0.50,
        y + 0.045,
        content,
        ha="center",
        va="center",
        fontsize=9.5,
        color=text_colour,
        linespacing=1.3
    )

box_positions = [0.82, 0.61, 0.40, 0.19]

add_framework_box(
    box_positions[0],
    "Estimated Local Need",
    "Overall IMD and seven deprivation domains\n"
    "Exploratory need = mean of seven domain percentiles\n"
    "Wellbeing indicators are contextual only"
)
add_framework_box(
    box_positions[1],
    "Actual Social Action Provision",
    "Influenced by local need, organisational capacity,\n"
    "volunteers, funding, leadership and other local services"
)

add_framework_box(
    box_positions[2],
    "Recorded Social Action Provision",
    "APFR social-action records captured in Caritas Westminster data\n"
    "Affected by reporting completeness and data quality"
)

add_framework_box(
    box_positions[3],
    "Observed Spatial Alignment",
    "Descriptive analysis, interactive mapping, correlation,\n"
    "ranking measures and analytical dashboard"
)

for upper_y, lower_y in zip(
    box_positions[:-1],
    box_positions[1:]
):
    arrow = FancyArrowPatch(
        (0.50, upper_y - 0.01),
        (0.50, lower_y + 0.145),
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.6,
        color=border_colour
    )

    ax.add_patch(arrow)

plt.tight_layout(pad=0.5)

plt.savefig(
    "conceptual_framework.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()